<a href="https://colab.research.google.com/github/makkergauri/ganga-water-quality/blob/main/02_station_registry.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Cell 1 - load the extracted rows (no PDFs needed from here on)
from google.colab import drive
drive.mount('/content/drive')
import re
import pandas as pd

base = "/content/drive/MyDrive/ganga_project"
raw = pd.read_csv(f"{base}/interim/nwmp_extracted_raw.csv", dtype=str).fillna("")   # keep everything as text
print(raw.shape, "| years:", sorted(raw["year"].unique()))


Mounted at /content/drive
(938, 27) | years: ['2016', '2017', '2018', '2019', '2020', '2021', '2022', '2024']


In [2]:
# Cell 2 - one row per station code, with a suggested type for me to review by hand
# Ganga table pages found in Cell 5 of notebook 01 (2017's real table was one page later: 12-17)
core_pages = {"2016": range(8, 11), "2017": range(12, 18), "2018": range(11, 18), "2019": range(12, 17),
              "2020": range(13, 17), "2021": range(20, 28), "2022": range(22, 31), "2024": range(9, 14)}

def full_name(r):
    # put wrapped name pieces back in order: [before] pieces + main name + [after] pieces
    before = " ".join(t.strip() for pos, t in re.findall(r"\[(before|after)\] ([^\[]+)", r["name_fragments"]) if pos == "before")
    after = " ".join(t.strip() for pos, t in re.findall(r"\[(before|after)\] ([^\[]+)", r["name_fragments"]) if pos == "after")
    return re.sub(r"\s+", " ", f"{before} {r['location_raw']} {after}").strip()

raw["full_name"] = raw.apply(full_name, axis=1)
raw["mentions_ganga"] = (raw["row_text"] + " " + raw["name_fragments"]).str.upper().str.contains("GANGA")
raw["on_core_page"] = raw.apply(lambda r: int(r["page"]) in core_pages[r["year"]], axis=1)
cand = raw[raw["mentions_ganga"] | raw["on_core_page"]]

def suggest(names, states):
    n, s = " ".join(names).upper(), " ".join(states).upper()
    if "CANAL" in n:
        return "canal"
    if "RAMGANGA" in n or "RAM GANGA" in n:
        return "other river"
    if ("BHAGIRATHI" in n or "ALAKNANDA" in n) and "UTTARAKHAND" in s:
        return "headstream"
    if "BHAGIRATHI" in n:
        return "check (Bhagirathi outside Uttarakhand)"
    if "GANGA" in n or "GANGES" in n:
        return "main stem"
    return "check (no GANGA in name)"

rows = []
for code, grp in cand.groupby("station_code"):
    names = sorted(set(grp["full_name"]) - {""})
    states = sorted(set(grp["state_raw"]) - {""})
    years = sorted(grp["year"].unique())
    rows.append({"station_code": code,
                 "suggested_type": suggest(names, states),
                 "final_type": "",            # I fill this in by hand: main stem / headstream / canal / other river / drop
                 "clean_name": "",            # I fill this in by hand (one clear name per station)
                 "state": " / ".join(states),
                 "n_years": len(years),
                 "years": ",".join(years),
                 "names_seen": " || ".join(names),
                 "notes": ""})

registry = pd.DataFrame(rows).sort_values(["suggested_type", "state", "station_code"])
registry.to_csv(f"{base}/interim/station_registry_draft.csv", index=False)
print(f"{len(registry)} candidate stations saved to interim/station_registry_draft.csv\n")
print("suggested types:")
print(registry["suggested_type"].value_counts().to_string())
print("\nhow many years each station appears in:")
print(registry["n_years"].value_counts().sort_index().to_string())
print("\nexamples:")
print(registry[["station_code", "suggested_type", "state", "n_years", "names_seen"]].head(12).to_string(index=False))

136 candidate stations saved to interim/station_registry_draft.csv

suggested types:
suggested_type
main stem                   104
check (no GANGA in name)     24
headstream                    6
canal                         2

how many years each station appears in:
n_years
1    32
2     3
3     3
4     3
5     3
6    14
7    21
8    57

examples:
station_code           suggested_type                    state  n_years                                                                                                                                                                                                                                                                                                                                                                                                            names_seen
        1061                    canal UTTARAKHAN / UTTARAKHAND        8 GANGA AT HARIDWAR D/S || GANGA AT HARIDWAR D/S, UPPER GANGA CANAL D/S BALKUMARI MANDIR, || GANGA A

In [3]:
# Cell 3 - pre-fill the review columns, and list the stations that need my attention most
def clean_state(s):
    s = s.upper()
    for key, name in [("UTTARAKH", "Uttarakhand"), ("UTTAR P", "Uttar Pradesh"), ("BIHAR", "Bihar"),
                      ("JHARKH", "Jharkhand"), ("WEST B", "West Bengal"), ("W.B", "West Bengal"),
                      ("HIMACHAL", "Himachal Pradesh"), ("DELHI", "Delhi"), ("HARYANA", "Haryana")]:
        if key in s:
            return name
    return s.title()

latest_name = (raw.sort_values("year").groupby("station_code")["full_name"]
               .agg(lambda s: [x for x in s if x][-1] if any(s) else ""))

registry["state"] = registry["state"].apply(lambda s: clean_state(s.split(" / ")[-1]) if s else "")
registry["final_type"] = registry["suggested_type"].where(~registry["suggested_type"].str.startswith("check"), "")
registry["clean_name"] = registry["station_code"].map(latest_name).str.title()
registry["same_as_code"] = ""
cols = ["station_code", "suggested_type", "final_type", "clean_name", "same_as_code", "state",
        "n_years", "years", "names_seen", "notes"]
registry = registry[cols]
registry.to_csv(f"{base}/interim/station_registry_for_review.csv", index=False)
print(f"saved interim/station_registry_for_review.csv ({len(registry)} stations)\n")

pd.set_option("display.max_colwidth", 90)
show = ["station_code", "state", "n_years", "years", "clean_name"]
print("A) suggested CANAL - is the sample taken IN the canal, or is the canal just a landmark?")
print(registry[registry["suggested_type"] == "canal"][show].to_string(index=False))
print("\nB) CHECK - usually another river from the table edges (-> drop), sometimes a Ganga station named after a town")
print(registry[registry["suggested_type"].str.startswith("check")][show].to_string(index=False))
print("\nC) HEADSTREAM (Bhagirathi / Alaknanda in Uttarakhand)")
print(registry[registry["suggested_type"] == "headstream"][show].to_string(index=False))
print("\nD) MAIN STEM stations present in 5 years or fewer - new stations, or a station whose code changed?")
print(registry[(registry["suggested_type"] == "main stem") & (registry["n_years"] <= 5)]
      .sort_values(["state", "clean_name"])[show].to_string(index=False))

saved interim/station_registry_for_review.csv (136 stations)

A) suggested CANAL - is the sample taken IN the canal, or is the canal just a landmark?
station_code       state  n_years                                   years                                                clean_name
        1061 Uttarakhand        8 2016,2017,2018,2019,2020,2021,2022,2024 River Ganga D/S At Balkumari Mandir, Ajeetpur, Haridwar D
        2727 Uttarakhand        6           2016,2018,2019,2021,2022,2024                Upper Ganga Canal D/S At Roorkee, Haridwar

B) CHECK - usually another river from the table edges (-> drop), sometimes a Ganga station named after a town
station_code            state  n_years                              years                                                          clean_name
        1553                         1                               2017                                       River Yamuna U/S Paonta Sahib
        4486 Himachal Pradesh        1                     

In [4]:
# Cell 4 - apply my review decisions, clean names/states, and search for hidden problems
decisions = {   # code: (final_type, clean_name or "" to keep the auto name, note)
    "1061": ("main stem", "Ganga at Haridwar D/S (Balkumari Mandir, Ajeetpur)", "canal only a landmark; river station"),
    "2727": ("canal", "Upper Ganga Canal D/S Roorkee", "canal station, not the river"),
    "1489": ("main stem", "Ganga at Devprayag (after Bhagirathi-Alaknanda confluence)", "first main-stem station"),
    "5451": ("headstream", "Mandakini at Kedarnath", "Mandakini joins the Alaknanda at Rudraprayag"),
    "1485": ("headstream", "Mandakini before confluence with Alaknanda, Rudraprayag", ""),
    "5452": ("headstream", "Alaknanda at Badrinath", ""),
    "20048": ("drop", "", "probably wastewater sampling point (verify in 2018 PDF)"),
    "20049": ("drop", "", "oxidation ponds = sewage treatment, not river water"),
    "20050": ("drop", "", "Jagjitpur = Haridwar STP site; probably wastewater (verify)"),
    "4432": ("drop", "", "Himachal stream (Dhauli Khad), not the Ganga"),
    "2555": ("main stem", "", "verify exact location (Punpun is also a tributary name)"),
    "30005": ("main stem", "", "CPCB inter-state boundary station; GPS in monthly reports"),
    "30006": ("main stem", "", "CPCB inter-state boundary station; GPS in monthly reports"),
    "30075": ("main stem", "", "CPCB inter-state boundary station; GPS in monthly reports"),
    "30076": ("main stem", "", "CPCB inter-state boundary station; GPS in monthly reports"),
}
for code in ["4486", "5248", "5252", "5255", "5766", "2626", "2627", "4431", "5251", "5254", "5256",
             "1553", "1490", "1492", "1493", "1494", "1306", "2913"]:
    decisions[code] = ("drop", "", "other river (table edge)")

reg = registry.copy()
for code, (ftype, name, note) in decisions.items():
    m = reg["station_code"] == code
    reg.loc[m, "final_type"] = ftype
    if name:
        reg.loc[m, "clean_name"] = name
    reg.loc[m, "notes"] = note

# tidy names: remove encoding junk (e.g. "Gangaâ") and trailing cut-off bits
reg["clean_name"] = (reg["clean_name"].str.replace(r"[^\x00-\x7F]+", " ", regex=True)
                     .str.replace(r"\s+(Pradesh|D|Dist)\.?\s*$", "", regex=True)
                     .str.replace(r"\s+", " ", regex=True).str.strip(" ,"))
reg.loc[reg["state"] == "Uttar", "state"] = "Uttar Pradesh"
reg.loc[reg["station_code"] == "30076", "state"] = "Bihar"

print("final types:")
print(reg["final_type"].replace("", "(still empty)").value_counts().to_string())

# check 1: possible code changes = same distinctive place word, years that don't overlap
common = {"RIVER", "GANGA", "AT", "U/S", "D/S", "B/C", "A/C", "DIST", "DIST.", "NEAR", "OF", "THE", "AND", "TO",
          "GHAT", "BRIDGE", "ROAD", "INTAKE", "POINT", "WATER", "UP", "U.P.", "BEFORE", "AFTER", "CONFLUENCE"}
def place_words(name):
    return {w for w in re.split(r"[^A-Z/.]+", name.upper()) if len(w) >= 5 and w not in common}
keep = reg[reg["final_type"].isin(["main stem", "headstream", "canal"])]
info = {r.station_code: (place_words(r.names_seen), set(r.years.split(","))) for r in keep.itertuples()}
print("\npossible code changes (shared place word, no overlapping years):")
found = False
codes = sorted(info)
for i, a in enumerate(codes):
    for b in codes[i + 1:]:
        shared = info[a][0] & info[b][0]
        if shared and not (info[a][1] & info[b][1]):
            found = True
            print(f"  {a} ({','.join(sorted(info[a][1]))})  <->  {b} ({','.join(sorted(info[b][1]))})  share: {', '.join(sorted(shared))}")
if not found:
    print("  none")

# check 2: suspicious words among main-stem stations
sus = reg[(reg["final_type"] == "main stem") &
          reg["names_seen"].str.upper().str.contains(r"CANAL|DRAIN|NALA|NALLAH|STP|POND|KHAD|RAMGANGA|YAMUNA|KALI|OUTLET")]
print(f"\nmain-stem stations with suspicious words ({len(sus)}):")
print(sus[["station_code", "state", "years", "clean_name"]].to_string(index=False) if len(sus) else "  none")

reg.to_csv(f"{base}/interim/station_registry_v1.csv", index=False)
print("\nsaved interim/station_registry_v1.csv")

final types:
final_type
main stem     105
drop           22
headstream      8
canal           1

possible code changes (shared place word, no overlapping years):
  none

main-stem stations with suspicious words (3):
station_code         state                                   years                                                                                                 clean_name
        1061   Uttarakhand 2016,2017,2018,2019,2020,2021,2022,2024                                                         Ganga at Haridwar D/S (Balkumari Mandir, Ajeetpur)
        1070 Uttar Pradesh 2016,2017,2018,2019,2020,2021,2022,2024 River Ganga At Varanasi U/S Near Vishwasundari Bridge B/C Drains( Nagwa Nala, Samne Ghat Nala,Nakhhi Nala)
        1335   West Bengal                2019,2020,2021,2022,2024                                                        River Ganga At Patikali Near Durga Chak West Bengal

saved interim/station_registry_v1.csv


In [5]:
# Cell 5 - before cleaning: which non-numeric values actually occur in the kept stations?
from collections import Counter

kept_codes = set(reg.loc[reg["final_type"].isin(["main stem", "headstream", "canal"]), "station_code"])
data = raw[raw["station_code"].isin(kept_codes)].copy()
value_cols = [c for c in data.columns if c.endswith("_min") or c.endswith("_max")]
print(f"{len(data)} station-year rows kept | {len(value_cols)} value columns")
print("duplicate (year, station) rows:", int(data.duplicated(["year", "station_code"]).sum()))

number = re.compile(r"-?\d+(\.\d+)?")
kinds = Counter()
odd = Counter()
for col in value_cols:
    for v in data[col]:
        v = str(v).strip()
        if v == "":
            kinds[(col, "blank")] += 1
        elif number.fullmatch(v.replace(",", "")):
            kinds[(col, "number")] += 1
        else:
            kinds[(col, "other")] += 1
            odd[(v, col.rsplit("_", 1)[0])] += 1

print("\nper parameter: numbers / blanks / other")
for prm in dict.fromkeys(c.rsplit("_", 1)[0] for c in value_cols):
    n = sum(kinds[(f"{prm}_{s}", "number")] for s in ["min", "max"])
    b = sum(kinds[(f"{prm}_{s}", "blank")] for s in ["min", "max"])
    o = sum(kinds[(f"{prm}_{s}", "other")] for s in ["min", "max"])
    print(f"  {prm:8s} numbers {n:4d} | blank {b:4d} | other {o:3d}")

print("\nall non-numeric values (value, parameter): count")
for (v, prm), n in odd.most_common(60):
    print(f"  {v!r:22s} {prm:8s} {n}")# Cell 5 - before cleaning: which non-numeric values actually occur in the kept stations?
from collections import Counter

kept_codes = set(reg.loc[reg["final_type"].isin(["main stem", "headstream", "canal"]), "station_code"])
data = raw[raw["station_code"].isin(kept_codes)].copy()
value_cols = [c for c in data.columns if c.endswith("_min") or c.endswith("_max")]
print(f"{len(data)} station-year rows kept | {len(value_cols)} value columns")
print("duplicate (year, station) rows:", int(data.duplicated(["year", "station_code"]).sum()))

number = re.compile(r"-?\d+(\.\d+)?")
kinds = Counter()
odd = Counter()
for col in value_cols:
    for v in data[col]:
        v = str(v).strip()
        if v == "":
            kinds[(col, "blank")] += 1
        elif number.fullmatch(v.replace(",", "")):
            kinds[(col, "number")] += 1
        else:
            kinds[(col, "other")] += 1
            odd[(v, col.rsplit("_", 1)[0])] += 1

print("\nper parameter: numbers / blanks / other")
for prm in dict.fromkeys(c.rsplit("_", 1)[0] for c in value_cols):
    n = sum(kinds[(f"{prm}_{s}", "number")] for s in ["min", "max"])
    b = sum(kinds[(f"{prm}_{s}", "blank")] for s in ["min", "max"])
    o = sum(kinds[(f"{prm}_{s}", "other")] for s in ["min", "max"])
    print(f"  {prm:8s} numbers {n:4d} | blank {b:4d} | other {o:3d}")

print("\nall non-numeric values (value, parameter): count")
for (v, prm), n in odd.most_common(60):
    print(f"  {v!r:22s} {prm:8s} {n}")

740 station-year rows kept | 18 value columns
duplicate (year, station) rows: 0

per parameter: numbers / blanks / other
  temp     numbers 1452 | blank    2 | other  26
  do       numbers 1480 | blank    0 | other   0
  ph       numbers 1478 | blank    0 | other   2
  cond     numbers 1415 | blank   38 | other  27
  bod      numbers 1440 | blank    2 | other  38
  nitrate  numbers 1024 | blank   90 | other 366
  fc       numbers 1365 | blank   45 | other  70
  tc       numbers 1391 | blank   29 | other  60
  fs       numbers  710 | blank  484 | other 286

all non-numeric values (value, parameter): count
  '-'                    fs       255
  '-'                    nitrate  170
  'BDL'                  nitrate  113
  '_'                    nitrate  44
  '__'                   nitrate  38
  'BDL'                  bod      36
  'BDL'                  fs       30
  '-'                    fc       24
  '-'                    tc       24
  '_'                    fc       22
  '-'          

In [6]:
# Cell 6 - clean the values into a tidy long table: raw text + number + qualifier + flags (nothing auto-fixed)
import numpy as np

NOT_REPORTED = {"", "-", "--", "_", "__"}
units = {"temp": "degC", "do": "mg/L", "ph": "", "cond": "umhos/cm", "bod": "mg/L",
         "nitrate": "mg/L", "fc": "MPN/100mL", "tc": "MPN/100mL", "fs": "MPN/100mL"}
plausible = {"temp": (0, 40), "do": (0, 20), "ph": (4, 11), "cond": (10, 5000), "bod": (0, 100),
             "nitrate": (0, 50), "fc": (0, 1e8), "tc": (0, 1e8), "fs": (0, 1e8)}

def parse(v):
    s = str(v).strip()
    if s in NOT_REPORTED:
        return np.nan, "not reported"
    if s.upper() == "BDL":
        return np.nan, "below detection limit"
    if number.fullmatch(s.replace(",", "")):
        return float(s.replace(",", "")), ""
    return np.nan, "unreadable - check PDF"

rows = []
for r in data.itertuples(index=False):
    r = r._asdict()
    for col in value_cols:
        prm, stat = col.rsplit("_", 1)
        if prm == "fs" and int(r["year"]) < 2019:
            continue                                   # fecal streptococci not measured before 2019
        val, qual = parse(r[col])
        rows.append({"year": int(r["year"]), "station_code": r["station_code"], "parameter": prm,
                     "stat": stat, "unit": units[prm], "value_raw": r[col], "value": val,
                     "qualifier": qual, "flag": "", "source_file": r["source_file"], "page": int(r["page"])})
long = pd.DataFrame(rows)

# flag 1: physically implausible values
for prm, (lo, hi) in plausible.items():
    m = (long["parameter"] == prm) & long["value"].notna() & ((long["value"] < lo) | (long["value"] > hi))
    long.loc[m, "flag"] += f"outside plausible range {lo}-{hi}; "

# flag 2: min larger than max (same station, year, parameter)
wide = long.pivot_table(index=["year", "station_code", "parameter"], columns="stat", values="value").reset_index()
bad = wide[wide["min"] > wide["max"]]
for b in bad.itertuples():
    m = (long["year"] == b.year) & (long["station_code"] == b.station_code) & (long["parameter"] == b.parameter)
    long.loc[m, "flag"] += "min > max; "

# flag 3: big jumps vs the same station's usual value (median over years)
for prm, ratio in [("cond", 3), ("bod", 5)]:
    sub = long[(long["parameter"] == prm) & long["value"].notna()]
    med = sub.groupby(["station_code", "stat"])["value"].transform("median")
    jump = sub[(med > 0) & ((sub["value"] >= ratio * med) | (sub["value"] <= med / ratio))]
    long.loc[jump.index, "flag"] += f"{ratio}x jump vs station's usual value; "

long["station_name"] = long["station_code"].map(reg.set_index("station_code")["clean_name"])
long.to_csv(f"{base}/interim/ganga_values_long.csv", index=False)

print(f"{len(long)} values saved to interim/ganga_values_long.csv\n")
print("qualifiers:")
print(long["qualifier"].replace("", "(number)").value_counts().to_string())
check = long[(long["flag"] != "") | (long["qualifier"] == "unreadable - check PDF")]
print(f"\nvalues to check by hand in the PDFs: {len(check)}")
pd.set_option("display.max_colwidth", 60)
print(check[["year", "page", "station_code", "station_name", "parameter", "stat", "value_raw", "qualifier", "flag"]]
      .sort_values(["year", "page"]).to_string(index=False))

12848 values saved to interim/ganga_values_long.csv

qualifiers:
qualifier
(number)                  11755
not reported                883
below detection limit       207
unreadable - check PDF        3

values to check by hand in the PDFs: 29
 year  page station_code                                                      station_name parameter stat value_raw              qualifier                                                                flag
 2016    10         1469        River Ganga At Diamond Harbour, 24 Parganas (S)West Bengal      cond  max     13370                                                          outside plausible range 10-5000; 
 2017    17         1052                      River Ganga At Uluberia , Howrah,West Bengal      cond  max      6482                        outside plausible range 10-5000; 3x jump vs station's usual value; 
 2017    17         1469        River Ganga At Diamond Harbour, 24 Parganas (S)West Bengal      cond  max      7526                    